# LEGO Sorter - Quick Experiments

Lightweight notebook for quick Blender experiments via MCP.

**Use this for**: Quick tests, scene inspection, parameter experimentation

**Use full pipeline notebook for**: Complete simulation runs

In [ ]:
# Quick Setup
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), "utils"))
from utils.blender_mcp_client import BlenderMCPClient

client = BlenderMCPClient(timeout=120)
print("✅ Ready" if client.test_connection() else "❌ Connection failed")

## Quick Actions

In [ ]:
# Get Scene Info
client.execute_code("""
import bpy
print(f"Objects: {len(bpy.data.objects)}")
print(f"Collections: {len(bpy.data.collections)}")
for col in bpy.data.collections:
    print(f"  - {col.name}: {len(col.objects)} objects")
""", "Scene Info")

In [ ]:
# Clear Scene
client.execute_script_file('blender/clear_scene.py', 'Clear')

In [ ]:
# Create Test Bucket
client.execute_script_file('blender/create_sorting_bucket.py', 'Bucket')

## Experimentation Area

Use this cell for quick Blender code experiments:

In [ ]:
# Your experiment code here
code = """
import bpy

# Example: List all mesh objects
meshes = [obj for obj in bpy.data.objects if obj.type == 'MESH']
print(f"Found {len(meshes)} mesh objects:")
for obj in meshes[:10]:  # Show first 10
    print(f"  - {obj.name}: {len(obj.data.vertices)} vertices")
"""

client.execute_code(code, "Experiment")

## Parameter Testing

Test different parameter values quickly:

In [ ]:
# Test different bucket sizes
sizes = [0.20, 0.24, 0.28]  # Different top sizes

for size in sizes:
    print(f"\n=== Testing bucket size: {size} ===")
    
    code = f"""
import bpy

# Clear previous
col = bpy.data.collections.get("test_bucket")
if col:
    for obj in col.objects:
        bpy.data.objects.remove(obj, do_unlink=True)
    bpy.data.collections.remove(col)

# Create test bucket
col = bpy.data.collections.new("test_bucket")
bpy.context.scene.collection.children.link(col)

bpy.ops.mesh.primitive_cube_add(size={size}, location=(0, 0, 0))
obj = bpy.context.active_object
if obj:
    obj.name = f"TestBucket_{{size}}"
    col.objects.link(obj)
    print(f"Created bucket with size {{size}}")
"""
    
    client.execute_code(code, f"Test Size {size}")

## Physics Debugging

In [ ]:
# Check physics setup
client.execute_code("""
import bpy

rigid_bodies = [obj for obj in bpy.data.objects if obj.rigid_body]
print(f"Rigid body objects: {len(rigid_bodies)}")

for obj in rigid_bodies[:5]:  # Show first 5
    rb = obj.rigid_body
    print(f"\n{obj.name}:")
    print(f"  Type: {rb.type}")
    print(f"  Mass: {rb.mass}")
    print(f"  Friction: {rb.friction}")
""", "Physics Check")

In [ ]:
# Check current frame state
client.execute_code("""
import bpy

scene = bpy.context.scene
print(f"Current frame: {scene.frame_current}")
print(f"Frame range: {scene.frame_start} - {scene.frame_end}")

# Sample object positions
for obj in list(bpy.data.objects)[:3]:
    loc = obj.location
    print(f"{obj.name}: ({loc.x:.3f}, {loc.y:.3f}, {loc.z:.3f})")
""", "Frame State")

## Conveyor Belt Transport Testing

In [ ]:
# Step 1: Clear and rebuild scene
print("=== Building Conveyor Scene ===\n")

# Clear scene
client.execute_script_file('blender/clear_scene.py', 'Clear Scene')

# Create bucket with hole
client.execute_script_file('blender/create_sorting_bucket.py', 'Create Bucket')

# Create conveyor belt with animated slats
client.execute_script_file('blender/create_conveyor_belt.py', 'Create Conveyor')

# Import LEGO parts
client.execute_script_file('blender/import_lego_parts.py', 'Import Parts')

# Setup physics
client.execute_script_file('blender/animate_lego_physics.py', 'Setup Physics')

print("\n✓ Scene built")

In [ ]:
# Step 2: Position parts ON the conveyor belt
client.execute_script_file('blender/place_parts_on_conveyor.py', 'Position Parts on Conveyor')

# Step 2b: Enhance physics for reliable transport
client.execute_script_file('blender/enhance_conveyor_physics.py', 'Enhance Physics')

In [ ]:
# Step 3: Check initial state at frame 1
client.execute_code("""
import bpy

scene = bpy.context.scene
scene.frame_set(1)

print("=== Frame 1 State ===")

# Check LEGO parts positions
lego_col = bpy.data.collections.get("lego_parts")
if lego_col:
    print(f"\\nLEGO Parts ({len(lego_col.objects)} objects):")
    for obj in list(lego_col.objects)[:5]:
        loc = obj.location
        print(f"  {obj.name}: ({loc.x:.3f}, {loc.y:.3f}, {loc.z:.3f})")

# Check conveyor belt position
conveyor = bpy.data.objects.get("Conveyor_Belt")
if conveyor:
    loc = conveyor.location
    print(f"\\nConveyor Belt: ({loc.x:.3f}, {loc.y:.3f}, {loc.z:.3f})")
    print(f"  Dimensions: {conveyor.dimensions.x:.3f} x {conveyor.dimensions.y:.3f} x {conveyor.dimensions.z:.3f}")

# Check first slat
slats = [obj for obj in bpy.data.objects if "Slat" in obj.name]
if slats:
    slat = slats[0]
    loc = slat.location
    print(f"\\nFirst Slat: ({loc.x:.3f}, {loc.y:.3f}, {loc.z:.3f})")
""", "Check Frame 1")

In [ ]:
# Step 4: Run simulation for 50 frames and check part movement
client.execute_code("""
import bpy

scene = bpy.context.scene
lego_col = bpy.data.collections.get("lego_parts")

if lego_col and len(lego_col.objects) > 0:
    test_part = list(lego_col.objects)[0]
    
    # Record positions at different frames
    positions = []
    
    for frame in [1, 10, 20, 30, 40, 50]:
        scene.frame_set(frame)
        bpy.context.view_layer.update()
        loc = test_part.location.copy()
        positions.append((frame, loc.x, loc.y, loc.z))
    
    print("=== Part Movement Analysis ===")
    print(f"\\nTracking: {test_part.name}")
    print("Frame | X       | Y       | Z       | ΔX from start")
    print("------|---------|---------|---------|---------------")
    
    start_x = positions[0][1]
    for frame, x, y, z in positions:
        delta_x = x - start_x
        print(f"{frame:5d} | {x:7.3f} | {y:7.3f} | {z:7.3f} | {delta_x:+7.3f}")
    
    # Check if part moved significantly along X axis (conveyor direction)
    total_movement = positions[-1][1] - positions[0][1]
    if abs(total_movement) > 0.05:
        print(f"\\n✓ Part moved {total_movement:.3f} units along conveyor!")
    else:
        print(f"\\n❌ Part barely moved ({total_movement:.3f} units) - friction issue?")
else:
    print("❌ No LEGO parts found")
""", "Analyze Movement")

In [ ]:
# Step 5: Take viewport screenshot to visualize current state
from utils.blender_mcp_client import BlenderMCPClient
import base64
from io import BytesIO
from PIL import Image
import json

# Execute the MCP tool to get viewport screenshot
result = client.execute_code("""
import json
# Return empty dict, we'll call MCP directly
print(json.dumps({}))
""", "Prepare for screenshot")

print("Taking viewport screenshot...")
# Note: This would require MCP Blender tool integration
# For now, use manual inspection in Blender viewport

## Summary

**Problem**: LEGO parts weren't being transported on the conveyor belt.

**Root Causes**:
1. Parts were positioned in bucket, not on conveyor surface
2. Friction values needed optimization
3. Initial contact between parts and slats wasn't guaranteed

**Solutions Implemented**:
1. **`place_parts_on_conveyor.py`** - Positions parts directly on belt surface in a grid
2. **`enhance_conveyor_physics.py`** - Increases friction (slats: 2.5, parts: 1.5), optimizes solver

**Expected Results**:
- Parts positioned on conveyor at start
- Slat friction (2.5) grips parts effectively
- Parts transported along inclined belt (upward motion)
- Solver iterations increased (20) for accurate friction simulation